In [7]:
import os
import sys
import pandas as pd
import numpy as np
# Sube un nivel desde 'notebook' y entra a 'src'
ruta_src = os.path.abspath(os.path.join("..", "src"))
if ruta_src not in sys.path:
    sys.path.append(ruta_src)
from feature_engineering import (build_quant_features,quant_features,)
from preprocessing import (temporal_split,fit_robust_params,transform_returns,)
input_train = pd.read_csv("../data/input_training.csv")
output_train = pd.read_csv("../data/output_training_gmEd6Zt.csv")

input_test = pd.read_csv("../data/input_test.csv")
output_test = pd.read_csv("../data/output_test_random.csv")

return_cols = [f"r{i}" for i in range(53)]

train = input_train.merge(
    output_train,
    on="ID",
    how="inner",
    validate="one_to_one"
)

train_dev, val_dev = temporal_split(train)
robust_params = fit_robust_params(train_dev,return_cols)
train_processed = transform_returns(train_dev,return_cols,robust_params)
val_processed = transform_returns(val_dev,return_cols,robust_params)

### Aqui tenemos:

$$train \longrightarrow \text{temporal split} \begin{cases} train\_dev \\ val\_dev \end{cases}$$

Donde `train_dev` y `val_dev` siguen siendo los retornos originales en bps.

Después:

$$train\_dev, val\_dev \longrightarrow \text{preprocessing 02} \begin{cases} train\_processed \\ val\_processed \end{cases}$$

Estos últimos contienen los r0...r52:

* **Robust-scaled** $\longrightarrow$ **clipped [-20,20]** $\longrightarrow$ **NaN=0**, además de las variables de missingness.

No se han construido las quant features de 03. Ese es el siguiente paso, pero se construyen desde `train_dev` y `val_dev`, no desde `train_processed`, porque `build_quant_features()` necesita los retornos en bps.


In [ ]:
train_features, val_features, lower_bounds, upper_bounds = (build_quant_features(
        train_dev,
        val_dev,
        return_cols
    ))

#train_processed / val_processed
# → secuencia r0...r52 preprocesada

# train_features / val_features
# → quant features construidas en bps